# Basic Pitch — Piano Transcription Test
Upload a WAV recording of piano playing and get back a MIDI file and visual piano roll.

**Run each cell in order.**

In [ ]:
# Cell 1 — Install dependencies (run once per session)
!pip install basic-pitch pretty-midi matplotlib --quiet

In [ ]:
# Cell 2 — Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import pretty_midi
from IPython.display import Audio, display
from google.colab import files
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

def midi_to_name(n):
    names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    return f"{names[n % 12]}{(n // 12) - 1}"

print('Ready.')

In [ ]:
# Cell 3 — Upload your WAV file
# Supported: mono or stereo WAV, any sample rate
print('Select your WAV file to upload:')
uploaded = files.upload()

audio_path = list(uploaded.keys())[0]
print(f'\nUploaded: {audio_path}')

# Play it back so you can confirm it sounds right
display(Audio(audio_path))

In [ ]:
# Cell 4 — Run Basic Pitch transcription
#
# onset_threshold:  higher = fewer notes detected, fewer false positives (0.0-1.0, default 0.5)
# frame_threshold:  higher = shorter notes dropped               (0.0-1.0, default 0.3)
# min_note_len:     minimum note duration in milliseconds        (default 58)
#
# If you're getting too many ghost notes, raise onset_threshold to 0.6 or 0.7
# If notes are being missed, lower it to 0.4

ONSET_THRESHOLD = 0.5
FRAME_THRESHOLD = 0.3
MIN_NOTE_LEN_MS  = 58

print('Transcribing...')
model_output, midi_data, note_events = predict(
    audio_path,
    ICASSP_2022_MODEL_PATH,
    onset_threshold=ONSET_THRESHOLD,
    frame_threshold=FRAME_THRESHOLD,
    minimum_note_length=MIN_NOTE_LEN_MS,
)
print(f'Done — {len(note_events)} notes detected.')

In [ ]:
# Cell 5 — Print detected notes as a table
print(f"{'#':>3}  {'Note':<5}  {'MIDI':>4}  {'Start':>8}  {'End':>8}  {'Duration':>9}  {'Confidence':>10}")
print('-' * 60)
for i, (start, end, pitch, amplitude, pitch_bend) in enumerate(note_events):
    duration = end - start
    print(f"{i+1:>3}  {midi_to_name(pitch):<5}  {pitch:>4}  {start:>7.3f}s  {end:>7.3f}s  {duration:>8.3f}s  {amplitude:>10.2f}")

In [ ]:
# Cell 6 — Piano roll visualisation
# Shows which notes were detected and when.
# Each horizontal bar = one note. Brighter = louder.

piano_roll = midi_data.get_piano_roll(fs=20)  # 20 frames per second
duration   = midi_data.get_end_time()

# Find the pitch range actually used (plus a little padding)
active_pitches = [int(pitch) for _, _, pitch, _, _ in note_events]
lo = max(21, min(active_pitches) - 3)
hi = min(108, max(active_pitches) + 3)

fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(
    piano_roll[lo:hi+1],
    aspect='auto',
    origin='lower',
    extent=[0, duration, lo, hi],
    cmap='Blues'
)

# Label every C note on the Y axis
c_notes = [n for n in range(lo, hi+1) if n % 12 == 0]
ax.set_yticks(c_notes)
ax.set_yticklabels([midi_to_name(n) for n in c_notes])

ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Note')
ax.set_title(f'Piano Roll — {os.path.basename(audio_path)}  ({len(note_events)} notes)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Compare detected notes against expected scale notes
# Edit EXPECTED_MIDI_NOTES to match what the student should have played.
# The script will flag any detected note not in the expected set.
#
# Example below: C major scale, right hand one octave (C4 to C5)
# MIDI: C4=60, D4=62, E4=64, F4=65, G4=67, A4=69, B4=71, C5=72

EXPECTED_MIDI_NOTES = {60, 62, 64, 65, 67, 69, 71, 72}   # <-- edit this

print(f"Expected notes: {sorted([midi_to_name(n) for n in EXPECTED_MIDI_NOTES])}")
print()
print(f"{'#':>3}  {'Note':<5}  {'Start':>8}  {'Status'}")
print('-' * 35)

wrong_count = 0
for i, (start, end, pitch, amplitude, pitch_bend) in enumerate(note_events):
    status = 'OK' if pitch in EXPECTED_MIDI_NOTES else '*** WRONG NOTE ***'
    if pitch not in EXPECTED_MIDI_NOTES:
        wrong_count += 1
    print(f"{i+1:>3}  {midi_to_name(pitch):<5}  {start:>7.3f}s  {status}")

print()
print(f'Wrong notes detected: {wrong_count} of {len(note_events)}')

In [ ]:
# Cell 8 — Save and download the MIDI file
midi_path = os.path.splitext(audio_path)[0] + '.mid'
midi_data.write(midi_path)
print(f'Saved: {midi_path}')
files.download(midi_path)